# E2B paper evidence replay
        Verify and analyze actual emulator/host observations. This notebook does not run a phone,
        manufacture benchmark data, or infer a memory-policy benefit from available RAM.
        Prior setup failures remain visible. Catalog and synthesized speech are explicit test inputs.
        An older voice case passed an overly permissive protocol-only final-answer check; the analyzer
        preserves its raw pass and reports the corrected valid count separately. A later run still failed.


In [ ]:
from pathlib import Path
import hashlib, json, os, platform, sys, zipfile
from datetime import datetime, timezone

# Set a full archive path if multiple matching datasets are attached.
WORKBENCH_ZIP = None
if WORKBENCH_ZIP is None:
    candidates = list(Path('/kaggle/input').rglob('kaggle_phase2_workbench.zip')) if Path('/kaggle/input').exists() else []
    if not candidates:
        candidates = list(Path.cwd().glob('output/kaggle_phase2_workbench.zip'))
    if len(candidates) != 1:
        raise RuntimeError('Attach the workbench dataset and set WORKBENCH_ZIP to its exact path')
    WORKBENCH_ZIP = candidates[0]
WORKBENCH_ZIP = Path(WORKBENCH_ZIP)
execution_environment = 'kaggle' if Path('/kaggle/working').exists() else 'local_not_kaggle'
run_base = Path('/kaggle/working') if execution_environment == 'kaggle' else Path.cwd() / 'scratch/kaggle_notebook_checks'
RUN = run_base / ('neuralos_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
ROOT = RUN / 'workbench'
ROOT.mkdir(parents=True)
with zipfile.ZipFile(WORKBENCH_ZIP) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)): raise ValueError('Duplicate archive members')
    for item in archive.infolist():
        target = (ROOT / item.filename).resolve()
        if item.is_dir() or not target.is_relative_to(ROOT.resolve()): raise ValueError('Invalid archive path')
        if (item.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('Symlink archive member')
    manifest = json.loads(archive.read('manifest.json'))
    if set(names) != {'manifest.json'} | {item['path'] for item in manifest['files']}: raise ValueError('Manifest/file mismatch')
    for item in manifest['files']:
        data = archive.read(item['path'])
        if hashlib.sha256(data).hexdigest() != item['sha256']: raise ValueError('Hash mismatch: ' + item['path'])
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
print('Verified archive:', hashlib.sha256(WORKBENCH_ZIP.read_bytes()).hexdigest())
print('Execution environment:', execution_environment)
print('Output directory:', RUN)


In [ ]:
from benchmarks.analyze_emulator_pilot import analyze
summary = analyze(ROOT / 'reports', RUN / 'paper_analysis', make_plot=True)
print(json.dumps(summary, indent=2))


In [ ]:
from benchmarks.analyze_native_residency import analyze as analyze_native
native_summary = analyze_native(ROOT / 'reports', RUN / 'native_analysis', make_plot=True)
print(json.dumps(native_summary, indent=2))
from benchmarks.analyze_native_editor import analyze as analyze_editor
editor_summary = analyze_editor(ROOT / 'reports', RUN / 'editor_analysis')
print(json.dumps(editor_summary, indent=2))
from benchmarks.analyze_real_world import analyze as analyze_real_world
real_summary = analyze_real_world(ROOT / 'benchmarks/real_world', RUN / 'real_world_analysis')
print(json.dumps(real_summary, indent=2))
from benchmarks.state_resident_metrics import analyze as analyze_state_metrics
state_summary = analyze_state_metrics(ROOT / 'benchmarks/real_world', RUN / 'real_world_analysis/state_resident_metrics.json')
print(json.dumps(state_summary, indent=2))
from benchmarks.profile_training_metrics import analyze as analyze_training
training_summary = analyze_training(ROOT / 'benchmarks/real_world', RUN / 'real_world_analysis/profile_training_metrics.json')
print(json.dumps(training_summary, indent=2))
from benchmarks.analyze_memory_deadline_comparison import analyze as analyze_comparison
for comparison in sorted((ROOT / 'benchmarks/real_world/comparisons').glob('*')):
    if (comparison / 'SHA256.json').is_file():
        print(json.dumps(analyze_comparison(comparison), indent=2))
from benchmarks.competition_readiness import audit
audit_summary = audit(ROOT, ROOT / 'benchmarks/real_world/comparisons/20261002T080929346071',
                      ROOT / 'benchmarks/real_world/profile_collections/20261002T073651969317')
(RUN / 'competition_evidence_audit.json').write_text(json.dumps(audit_summary, indent=2))
print(json.dumps(audit_summary, indent=2))


## Interpretation
        Task completion is checked against observed fixture state. Canvas OCR and audio transcription
        are separate tests. Small pilot counts do not establish general reliability. Choreographer
        callback cadence is not a complete rendered-frame or LMKD trace. Guest process PSS excludes
        the host Ollama model in the older pilot. The native analysis separately replays Android 17
        LiteRT-LM checks, actual Settings interaction and opt-in residency mechanism pilots. Its
        HWUI completion lateness is not display/input latency, and shell tracing is test authority.
        Failures, mixed versions and missing telemetry remain visible. These short, illustratively
        priced sessions do not establish a policy effect or general task reliability. Real VLC/game
        no-AI and native episodes are replayed separately; policy outcomes come from the retained
        comparison reports, including failed attempts. Native S24 FE and repeated held-out
        studies remain required. Public own-budget readings and empirical peak/deadline admission
        are implemented; calibrated forecasting and production benefits remain unproven. Observer 3
        records durable attempt starts/completions to expose interruptions. Literal-copy
        diagnostics execute no tools and are separate from real Settings tasks. The latest bound
        schema-repair episode passed actual typing/readback; preceding failures remain retained.

        For the paper track, keep the writeup within 3,000 words and connect the contribution to
        agentic software engineering. A shopping/typing demonstration is a functional prerequisite,
        not evidence of coding gains. See the bundled experiment protocol and working draft.
        RESEARCH_ROADMAP.md sets out the proposed joint policy and close prior art; it reports no novel
        algorithm result or superiority over those systems.


In [ ]:
from IPython.display import display, Image
figure = RUN / 'paper_analysis/emulator_pilot.png'
if figure.exists(): display(Image(filename=str(figure)))
native_figure = RUN / 'native_analysis/idle_gap_mechanism.png'
if native_figure.exists(): display(Image(filename=str(native_figure)))
writeup = (ROOT / 'paper/kaggle_writeup.md').read_text(encoding='utf-8')
print('Working writeup whitespace word count:', len(writeup.split()))
print('Analysis artifacts:', [str(p) for p in (RUN / 'paper_analysis').glob('*')])
